# C10-competition-craft — Session 1: The Contract

*One class session, roughly 80 minutes. Prerequisites:
C4-classical-ml-practice (the CSV → `(X, y)` bridge, kNN, scaling with
train statistics, pipelines, honest validation) and, through it, C1's
train/test discipline and metric family.*

**This session:** everything you built in C4 becomes a *submission*.
A competition (and the exam's applied problem) does not grade your
reasoning live — it runs your notebook, top to bottom, on a machine you
never touch, and then calls **one function you promised to define** on
rows you have never seen.
That promise is a *contract*: a fixed function name, a fixed input
form, a fixed output form.
Honor it and your model's quality decides your score; break it — wrong
return type, wrong length, wrong index — and the score is **zero**, no
matter how good the model was.
This session teaches the contract device this course uses
(`predict_labels`), the hidden-test protocol around it, and the
violations that turn good models into zeros.

Try every checkpoint yourself before reading on.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

## 1. The Graded-Notebook Model

**Motivation.**
C4 ended with a pointer: the competition setting adds a *hidden* test
set and a required prediction-function deliverable.
Here is the full picture of that setting — the **graded-notebook
model**:

1. You receive a labeled training table and a task statement.
2. You submit **one notebook**.
3. The grader runs it **top to bottom, in order, on a fresh machine**
   ("Restart & Run All" — nothing you ran interactively survives).
4. Your notebook must define **one prediction function with a stated
   signature**; the grader calls it on a **held-back** feature table
   your notebook has never read.
5. Your score combines three things: the notebook *runs clean*, the
   prediction function *honors its contract*, and the predictions
   *score well* on the held-back rows — plus, on the real exam, the
   quality of your written reasoning (Session 3's subject).
   Code *style* is explicitly not graded; code *discipline* is what
   the run-clean requirement grades.

Two consequences run through this whole unit:

- **The grader is a machine.**
  It will not squint at your almost-right output and award partial
  credit; a `list` where a `Series` was promised scores like an empty
  cell.
- **Every number you trust must be computable without the held-back
  rows.**
  The only honest estimate of your score is a validation split you
  carve from the training data yourself (Session 2's subject).

**Allowed imports in this unit** (mirroring the exam's applied-problem
register): `sklearn`, `numpy`, `pandas`, `matplotlib` — and the model
family is restricted to k-nearest neighbors, exactly as C4 taught.

### Checkpoint 1

1. Your notebook works perfectly when you run cells in the order you
   *wrote* them during exploration: 1, 2, 5, 3, 4.
   What happens at grading time, and which step of the
   graded-notebook model catches it?
2. Name the three machine-graded components of the score, and state
   which of them a brilliant model with a broken return type still
   earns.
3. Why can't you compute your own final score before submitting?
   What is the best substitute?

## 2. The Task and the Harness

**The unit's running competition.**
An apiary network wants to know, at the autumn inspection, which
honeybee colonies will **thrive** through winter and which will
**struggle** — so keepers can intervene (feed, treat, insulate) before
the cold.
Each row is one colony; the 12 numeric features are autumn sensor and
inspection readings; the label `outcome` is what the spring inspection
found.

**The harness is open-source on purpose.**
The training table `../data/train.csv` (600 colonies) was produced by
the seeded script `../data/make_dataset.py`, which you can read.
The same script also produced a **held-back split** of 200 further
colonies — that split is the grading set, and the *protocol* (Section
4) is that your notebook never touches it.

First look, in C4's ritual:

In [ ]:
df = pd.read_csv("../data/train.csv")

FEATURES = [c for c in df.columns if c != "outcome"]

print("shape:", df.shape)
print("columns:", list(df.columns))
print()
print(df["outcome"].value_counts())
print()
print("class fractions:", (df["outcome"].value_counts(normalize=True)).round(3).to_dict())

In [ ]:
# The bridge (C4): features as a DataFrame X, labels as an array y.
# We keep X as a DataFrame in this unit -- the contract's input is a
# DataFrame, and sklearn accepts it directly.
X = df[FEATURES]
y = df["outcome"].to_numpy()

print("X:", X.shape, "| y:", y.shape, "| labels:", np.unique(y))
print()
print("feature spreads (std), smallest and largest three:")
spreads = X.std().sort_values()
print(spreads.head(3).round(2))
print(spreads.tail(3).round(2))

Read the two printouts like a competitor:

- **Class imbalance, roughly 2:1.**
  About 63% of colonies thrive.
  A do-nothing rule ("every colony thrives") is 63% accurate — C1's
  majority-baseline warning applies, and Session 2 will grade with a
  metric built to punish exactly that rule.
- **Wildly different scales.**
  `varroa_mite_index` has spread ~1.3 while `apiary_elevation_m` has
  spread ~120: raw Euclidean distances would be dominated by
  large-scale columns — C4's scaling distortion, alive and well here.

### Checkpoint 2

1. Roughly what accuracy does the always-`thrives` rule earn on this
   table, and why is that number a warning rather than an
   achievement?
2. Without running anything: name the two features you most expect to
   dominate *raw* Euclidean distances, and the C4 fix.
3. The generator script is public and seeded.
   Does reading it help you predict better?
   Does *running* it with its grading flag and reading the held-back
   rows?
   (One sentence each — Section 4 formalizes the second answer.)

## 3. The Prediction-Function Contract

**Definition.**
This course's pinned contract device is a function named
**`predict_labels`**:

```python
def predict_labels(X_test):
    # X_test: pd.DataFrame with exactly the training feature columns
    #         (same names, same order), ANY number of rows, unlabeled.
    # returns: pd.Series of predicted labels.
    ...
```

The contract has five clauses.
The grader checks them mechanically, and **any violation scores
zero**:

| # | Clause | Typical violation |
|---|--------|-------------------|
| R1 | a callable named exactly `predict_labels` exists after the notebook runs | typo'd name; function defined inside an `if` that didn't run |
| R2 | it accepts a feature `DataFrame` of **any** length | hardcoding 200 rows; indexing columns by position that don't exist |
| R3 | it returns a **`pd.Series`** | returning a `list`, `np.ndarray`, or `DataFrame` |
| R4 | the result has `len(result) == len(X_test)` **and** `result.index.equals(X_test.index)` — one prediction per input row, in the input's order | dropping rows; `reset_index`; sorting |
| R5 | every value is a label from the training vocabulary | returning 0/1 codes when the labels are strings |

R4 deserves a beat: the grader aligns *your* Series with *its* answer
key **by index**.
The index is the row's identity; break it and every prediction may be
compared against the wrong colony.

**A minimal legal submission.**
The model is deliberately modest (scaled 5-NN, C4's default toolkit);
the point is the packaging: fit once, then let `predict_labels` do
*only* prediction:

In [ ]:
# Fit the whole recipe on ALL training rows (the submission's final model).
pipe = Pipeline([("scaler", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=5))])
pipe.fit(X, y)


def predict_labels(X_test):
    # Only prediction happens here: the pipeline was fitted above, and
    # its stored TRAIN statistics scale whatever rows arrive.
    return pd.Series(pipe.predict(X_test), index=X_test.index)


print(type(predict_labels))

In [ ]:
# Contract self-checks -- run them on a probe BEFORE submitting.
# The probe is just training rows; any rows with the right columns do.
probe = X.iloc[:25]
out = predict_labels(probe)

checks = {
    "R3 returns a Series   ": isinstance(out, pd.Series),
    "R4 length matches     ": len(out) == len(probe),
    "R4 index matches      ": out.index.equals(probe.index),
    "R5 labels from vocab  ": set(out.unique()) <= set(np.unique(y)),
}
for name, ok in checks.items():
    print(name, ok)
print()
print("first five predictions:")
print(out.head())

All four checks print `True`, and the head shows string labels
indexed 0–4 — the probe's own index, in the probe's own order.
This four-check cell is the single highest-value habit in this unit:
it costs seconds and catches every zero-scoring packaging bug before
the grader does.

### Checkpoint 3

1. Recite the five clauses from memory, each with one violation that
   trips it.
2. Why does the fitted pipeline live *outside* `predict_labels`
   instead of being fit inside it?
   (Two reasons: one about what `X_test` lacks, one about wasted
   work.)
3. A teammate's function passes all four self-checks on
   `probe = X.iloc[:25]` but crashes on `X.iloc[:37]`.
   Which clause was violated, and what bug pattern do you suspect?

## 4. The Hidden-Test Protocol, Honestly

**The protocol.**
The 200 held-back colonies exist right now, deterministically: the
seeded generator can rebuild them at any time.
"Hidden" is therefore a *protocol*, not an encryption scheme — and
this course states it honestly, once, here:

> **The held-back split is hidden from you, the student register — not
> from the graders.**
> At grading time, the course's grading notebooks regenerate the
> held-back rows with the same seeded script (its grading flag exists
> for exactly this) and score your `predict_labels` on them.
> Your notebook must run start to finish **without ever reading or
> printing the held-back rows**, and everything you report must be
> computable without them.
> An automated sweep checks student notebooks for exactly that.

**What the grader runs** (the pattern, shown — this cell is *theirs*,
never yours):

```python
# GRADER REGISTER -- executed at grading time, not in your notebook:
#   X_hidden, y_hidden = <the regenerated held-back features / labels>
preds = predict_labels(X_hidden)          # your contract, their rows
score = f1_score(y_hidden, preds, average="macro")
```

**Why the protocol has teeth even though you *could* peek.**
On the real exam the held-back rows are genuinely unavailable, so the
habit you are building — *never touch, never need* — is the exam
skill.
Practicing on a harness where peeking is physically possible but
forbidden is exactly how the discipline becomes automatic.
And practically: any number you compute from held-back rows is a lie
about your generalization — the same lie C4 called leakage, scaled up
from "a scaler saw test rows" to "you saw the answer key."

### Checkpoint 4

1. In one sentence each: what does "hidden" mean here for (i) you,
   (ii) the graders, (iii) the CI system that checks this course's
   own materials?
2. The grader's cell calls `predict_labels(X_hidden)` where
   `X_hidden` has 200 rows and your probe had 25.
   Which contract clause guarantees this just works?
3. Give two distinct reasons the never-touch rule survives even
   though the split is deterministically regenerable.

## 5. Violations Are Zeros: Three Postmortems

Each of the following submissions contains a *working model*.
Each scores zero.
Watch the checks fail — mechanically, the way the grader would.

**Postmortem A — wrong return type (breaks R3).**
`pipe.predict` returns a NumPy array; forgetting to wrap it is the
single most common packaging bug:

In [ ]:
def predict_labels_A(X_test):
    return pipe.predict(X_test)          # BROKEN: ndarray, not Series


out_A = predict_labels_A(probe)
print("type:", type(out_A).__name__)
print("R3 returns a Series:", isinstance(out_A, pd.Series))
print("...and an ndarray has no index for R4 to check at all.")

**Postmortem B — wrong length (breaks R4).**
"Cleaning" the grader's input.
This function drops rows it considers outliers — but the grader needs
a prediction for *every* row it asked about:

In [ ]:
def predict_labels_B(X_test):
    kept = X_test[X_test["varroa_mite_index"] < 4.0]   # BROKEN: drops rows
    return pd.Series(pipe.predict(kept), index=kept.index)


out_B = predict_labels_B(probe)
print("asked about:", len(probe), "rows | answered:", len(out_B))
print("R4 length matches:", len(out_B) == len(probe))

**Postmortem C — wrong index (breaks R4).**
The predictions are all present, even in the right order — but
`reset_index` replaced the rows' identities with 0..24, and the
grader aligns by index:

In [ ]:
def predict_labels_C(X_test):
    out = pd.Series(pipe.predict(X_test), index=X_test.index)
    return out.reset_index(drop=True)    # BROKEN: identities destroyed


probe_shifted = X.iloc[100:125]          # a probe whose index is NOT 0..24
out_C = predict_labels_C(probe_shifted)
print("probe index head:", list(probe_shifted.index[:3]))
print("output index head:", list(out_C.index[:3]))
print("R4 index matches:", out_C.index.equals(probe_shifted.index))

Note the trap inside the trap: on `probe = X.iloc[:25]`, whose index
*happens* to be 0..24, Postmortem C passes every check — and then
zeros on any other slice.
That is why the self-check probe should be a slice from the *middle*
of the table, not the top.

**Ordering is the same story.**
A function that sorts `X_test` (say, by `honey_stores_kg`) before
predicting and returns the sorted Series still carries the original
index labels — pandas alignment can rescue a grader that aligns by
index, but a grader comparing positionally is fully entitled to by
R4's "in the input's order" clause.
Never reorder; predict rows as they arrive.

### Checkpoint 5

1. For each postmortem A–C, name the clause broken and the one-line
   fix.
2. Why did Postmortem C pass all checks on `X.iloc[:25]` but fail on
   `X.iloc[100:125]`?
   What does that teach about choosing probes?
3. Write (on paper) a fourth postmortem: a function that breaks R5
   without breaking R3 or R4.

## 6. Leakage Traps, Competition Edition

C4 taught the two classic leaks; the competition setting adds a third.
All three produce *plausible* numbers and *no* error message:

- **Trap 1 — scaler fitted on training + validation rows.**
  Your validation score is then measured in coordinates the
  validation rows helped choose: gently, unquantifiably optimistic.
  Fix: the pipeline refits the scaler inside whatever rows it is
  `fit` on — keep preprocessing inside the pipeline, always.
- **Trap 2 — repeated peeking at one validation split.**
  Every model change you accept *because validation went up* spends a
  little of the split's honesty (Session 2 measures this precisely).
- **Trap 3 — touching the held-back rows at all.**
  The competition-specific sin: Section 4's protocol, and an
  automatic zero here.

Trap 1, demonstrated on a validation split (the numbers are close —
that is exactly why the bug survives code review; the *protocol* is
what breaks):

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)

honest = StandardScaler().fit(X_tr)                 # train rows only
leaky = StandardScaler().fit(pd.concat([X_tr, X_val]))   # BROKEN: saw val rows

print("honest mean of honey_stores_kg:", round(honest.mean_[0], 4))
print("leaky  mean of honey_stores_kg:", round(leaky.mean_[0], 4))
print("close -- but the leaky scaler let 150 evaluation rows shape")
print("the coordinate system they are then evaluated in.")

### Checkpoint 6

1. Rank the three traps by *size of score distortion* and separately
   by *severity of protocol violation*.
   Why do the rankings differ?
2. Which single C4 object makes Trap 1 structurally impossible, and
   what exactly does it do differently?
3. True or false, one sentence: "Trap 3 only matters if the
   held-back rows actually change my model."


## 7. Worked Exam-Style Example: a Complete Legal Submission

The full register — pinned identifiers, a validation number, the
contract, the bans.

> **Task.**
> For the apiary table `../data/train.csv` (600 rows, 12 features,
> labels `thrives`/`struggles`):
> **(a)** bridge to `X` (feature `DataFrame`, pinned column order) and
> `y`;
> **(b)** carve a seeded validation split:
> `train_test_split(X, y, test_size=150, random_state=SEED,
> stratify=y)`;
> **(c)** fit a `Pipeline` (`"scaler"` = `StandardScaler()`, `"knn"` =
> `KNeighborsClassifier(n_neighbors=5)`) on the training part and
> report `val_acc`, its validation accuracy;
> **(d)** refit the same pipeline on **all 600 rows** and define
> `predict_labels` to the contract;
> **(e)** run the four contract self-checks on
> `probe = X.iloc[200:230]`.
> **Constraints (zero points): any supervised estimator other than
> `KNeighborsClassifier` — including the LogisticRegression /
> RandomForest / GradientBoosting / SVC / MLP families, or
> re-implementing any of them by hand; reading the held-back split.
> Any preprocessing is allowed.**

**Solution, narrated.**
(a)–(b) are C4 verbatim; (c) is the estimator API; (d) is the one
competition-specific move — *the deliverable model is refit on
everything*, because every labeled row you leave out of the final fit
is signal the grader's rows never benefit from; (e) is Section 3's
habit, on a mid-table probe (Section 5's lesson).

In [ ]:
# (a) -- X, y, FEATURES exist from Section 2; restated for the register:
X = df[FEATURES]
y = df["outcome"].to_numpy()

# (b) seeded, stratified validation carve
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)

# (c) scaled 5-NN, judged on the held-out-from-fitting rows
sub = Pipeline([("scaler", StandardScaler()),
                ("knn", KNeighborsClassifier(n_neighbors=5))])
sub.fit(X_tr, y_tr)
val_acc = sub.score(X_val, y_val)
print("val_acc =", round(val_acc, 4))

In [ ]:
# (d) the final model sees every labeled row, then the contract wraps it
sub.fit(X, y)


def predict_labels(X_test):
    return pd.Series(sub.predict(X_test), index=X_test.index)


# (e) self-checks on a MID-TABLE probe
probe = X.iloc[200:230]
out = predict_labels(probe)
print("R3 Series :", isinstance(out, pd.Series))
print("R4 length :", len(out) == len(probe))
print("R4 index  :", out.index.equals(probe.index))
print("R5 vocab  :", set(out.unique()) <= set(np.unique(y)))

Grader's-eye view: the deliverables are `val_acc` and a
contract-honoring `predict_labels`; the instant-zero mistakes are a
banned estimator anywhere in the notebook, or any read of the
held-back split.
One number to carry into Session 2: `val_acc ≈ 0.787` *overstates*
how good this model is on the minority class — accuracy is the wrong
scoreboard for a 2:1 task, and fixing that is the next session's
whole subject.

### Checkpoint 7

1. Why refit on all 600 rows in (d) when the validation score in (c)
   was measured on a model fitted on only 450?
   What, exactly, is `val_acc` now an estimate *of*?
2. The ban clause names five model families *and* hand
   re-implementation.
   Why must the closer ("re-implementing them by hand") be there for
   the ban to mean anything?
3. After (d), is `val_acc` still an honest number to report?
   (Careful: the *model reported on* and the *model submitted* now
   differ — say precisely what a careful writeup would state.)

## 8. Common Pitfalls I

**Pitfall 1 — stale globals under Restart & Run All.**
`predict_labels` closes over whatever the surrounding notebook last
assigned.
If cell 12 refits `pipe` with a different `k` *after* the function
was defined, the submitted function silently uses the new fit — or,
worse, the exploratory fit only *you* had in memory vanishes on the
grader's fresh run.
Fix: one clearly-marked FINAL MODEL cell that fits the deliverable,
immediately followed by the contract cell — and nothing below them
touches those names (Session 3 makes this a checklist item).

**Pitfall 2 — assuming the held-back row count.**
`np.empty(200)`-style buffers or `range(200)` loops break R2 the
moment the grader probes with a different length.
Everything in this unit is already length-agnostic — keep it that
way.

**Pitfall 3 — answering in the wrong vocabulary.**
Mapping labels to 0/1 for convenience and forgetting to map back
breaks R5:

In [ ]:
def predict_labels_D(X_test):
    codes = (sub.predict(X_test) == "thrives").astype(int)   # BROKEN: 0/1
    return pd.Series(codes, index=X_test.index)


out_D = predict_labels_D(probe)
print("returned values:", sorted(pd.unique(out_D)))
print("R5 vocab:", set(out_D.unique()) <= set(np.unique(y)))

Fix: if you encode, decode at the boundary — the contract speaks the
training table's language.

**Pitfall 4 — self-checks on a too-convenient probe.**
Section 5's Postmortem C passed on `X.iloc[:25]` and zeroed on
everything else.
A good probe: mid-table slice, ≥20 rows, index not equal to
`0..n-1` — and run the checks *after* the final model cell, so they
test what the grader receives.

### Checkpoint 8

1. Draw (on paper) the cell order of a submission notebook that
   dodges Pitfall 1.
   Where do exploration cells go?
2. Which contract clause does each pitfall 1–4 threaten?
   (Pitfall 1 can threaten two — name both.)
3. Your final model cell and your k-sweep exploration disagree about
   `k`.
   Which one is the submission, and how would the grader find out?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. The grader runs cells strictly top to bottom on a fresh kernel;
   cell 3 will run before cell 5, using stale or missing names, and
   the notebook fails the "runs clean" component (step 3 of the
   model).
2. (i) The notebook runs clean top to bottom; (ii) the prediction
   function honors its contract; (iii) the predictions score well on
   the held-back rows.
   A brilliant model with a broken return type earns only (i).
3. The score is computed on held-back rows your notebook must never
   read.
   The substitute: a seeded validation split carved from the training
   table — the only honest signal (Session 2).

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. About 0.63 (the `thrives` fraction).
   It is the score of a rule that ignores every feature — any real
   submission must be judged against it, not against zero.
2. `apiary_elevation_m` and `distance_to_water_m` (spreads ~120 and
   ~90 dwarf every other column); the fix is standardization with
   train statistics — C4's `StandardScaler` inside the pipeline.
3. Reading the script helps you understand the *schema* and the
   honest protocol (and is encouraged); running it with the grading
   flag and reading the held-back rows is a protocol violation that
   zeroes the exercise — hidden means hidden from your notebook and
   your decisions, not undiscoverable.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. R1 exact callable name (typo; conditional definition); R2 any
   length (hardcoded row counts); R3 `pd.Series` (list/array/frame);
   R4 length and index equal to the input's (dropping rows,
   `reset_index`, sorting); R5 training vocabulary (0/1 codes for
   string labels).
2. `X_test` arrives unlabeled, so nothing could be fit on it anyway
   (no `y`); and fitting is the expensive, once-per-submission step —
   the function is called on arbitrary batches and must only
   predict.
3. R2.
   Something in the function depends on the probe's specific length
   or index — a hardcoded size, positional indexing past the end, or
   a buffer allocated for 25 rows.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. (i) For you: never read, never print, never let them influence
   anything.
   (ii) For the graders: regenerable on demand by the seeded script's
   grading flag, used once to score `predict_labels`.
   (iii) For CI: fully executable — the course's own checks
   regenerate and score deterministically, so "hidden" never means
   "unverifiable".
2. R2 — the function accepts a feature DataFrame of any length.
3. (i) The exam's held-back rows really are unreachable, so the
   habit is the transferable skill; (ii) any number influenced by
   held-back rows stops estimating generalization — it is leakage at
   maximum strength, an answer-key consultation.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. A: R3; wrap in `pd.Series(..., index=X_test.index)`.
   B: R4 (length); predict every row — never filter the grader's
   input.
   C: R4 (index); return with `index=X_test.index` and never
   `reset_index`.
2. The top-of-table probe's index (0..24) coincides with the default
   `RangeIndex` that `reset_index` installs, so the equality check
   passed by luck.
   Probes should have a *distinguishing* index — mid-table slices.
3. Any function returning a correctly-indexed Series of values
   outside {`thrives`, `struggles`} — e.g. returning `"THRIVES"`
   (case changed), `"ok"`, or 0/1 integers.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Distortion: Trap 3 ≫ Trap 2 > Trap 1 (a scaler leaks two numbers
   per column; peeking accumulates; reading the answers is unbounded).
   Violation severity: Trap 3 is an automatic zero; Traps 1–2 are
   dishonest numbers, not zeros.
   They differ because one is a *contract/protocol* breach and the
   others degrade an *estimate* — the grader can only mechanically
   police the first.
2. The `Pipeline`: its `fit` fits the scaler on exactly the rows
   being fit on, and its `predict`/`score` reuse stored statistics,
   so validation rows can never reach the scaler's `fit`.
3. False — the violation is reading them at all; whether they changed
   your model is unknowable from outside, which is exactly why the
   rule is absolute.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. The 450-row fit's validation score estimates the *recipe's*
   quality; refitting the same recipe on all 600 rows can only add
   information.
   `val_acc` is an estimate (typically slightly conservative) of the
   submitted model's held-back performance — an estimate of the
   recipe, measured on the smaller fit.
2. Without the closer, "use only kNN" is defeated by pasting a
   from-scratch logistic regression and claiming no banned *import*
   was used; the ban is on the model *family*, however expressed.
3. Yes, with one honest sentence: "validation score measured under
   the split protocol on a 450-row fit of the same recipe; the
   submitted model is that recipe refit on all rows."
   Claiming it as the submitted model's measured score would be
   sloppy; claiming a *held-back* score would be impossible.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Exploration cells first (loading, plots, sweeps), then one FINAL
   MODEL cell (fit on all rows), then the contract cell defining
   `predict_labels`, then the self-check cell — and no cell below
   ever rebinds `sub`/`pipe` or `predict_labels`.
2. Pitfall 1: R1 (the name may not exist or not survive a fresh run)
   and R4/R5 indirectly via whichever stale model answers; Pitfall
   2: R2; Pitfall 3: R5; Pitfall 4: no clause directly — it defeats
   your *detection* of R4 violations.
3. The final model cell — the grader reruns everything in order, so
   whatever that cell fits last is what `predict_labels` uses; the
   grader "finds out" nothing — it simply scores the notebook you
   actually submitted, not the one you remember running.

</details>